# Level Up 
## Building game agents that teach themselves with reinforcement learning

**Session by Brian Kayongo**

Most game AI is scripted: *if the player is close, attack*. Today we build agents that **get better on their own**, from experience.

Every one of them runs the same loop:

**ACT** → **FEEDBACK** (reward) → **LEARN** → **UPDATE** → repeat

| Level | You build | Big idea |
|---|---|---|
| 1 · The Loot Chests | An agent that finds the best treasure chest | Explore vs exploit |
| 2 · Dungeon Escape | A hero that learns to avoid lava | Delayed rewards |
| 3 · The Self-Play Champion | A tic-tac-toe brain that trains by playing itself | Self-play, the idea behind AlphaZero |
| 4 · The Wizard's Spellbook | A tiny language model trained with GRPO | How reasoning models are trained, and reward hacking |
| 5 · The Game-Dev Agent | An AI that writes game code and learns from failed tests | Improving without retraining |
| 6 · Boss Level *(optional, GPU)* | GRPO on a real LLM | The same recipe at real scale |

Run each cell with **Shift + Enter**.

---
# Level 1 · The Loot Chests 

The dungeon has three treasure chests. Each one has a hidden chance of dropping **rare loot**. Your agent gets 2,000 chest openings. It never sees the drop rates; it only sees what came out.

**Vocabulary**
- **Agent**: the player making choices
- **Action**: which chest to open
- **Reward**: 1 for rare loot, 0 for junk
- **Policy**: the agent's strategy

**The big tension: explore vs exploit.** Keep opening the chest that's been best so far (exploit), or try others in case they're better (explore)?

### 1a · Epsilon-greedy
Mostly open the best chest so far. With probability ε (epsilon), open a random one.

In [1]:
import numpy as np
rng = np.random.default_rng(42)

# Three treasure chests in the dungeon. Each has a hidden chance of dropping rare loot.
CHESTS = ["Wooden chest", "Iron chest", "Golden chest"]
TRUE_DROP_RATES = np.array([0.20, 0.35, 0.50])     # hidden from the agent. Golden is secretly best.

def open_chest(i):
    return int(rng.random() < TRUE_DROP_RATES[i])   # 1 = rare loot, 0 = junk

def epsilon_greedy(n_opens=2000, epsilon=0.1):
    counts = np.zeros(3); values = np.zeros(3); loot = 0
    for t in range(n_opens):
        if rng.random() < epsilon:
            a = int(rng.integers(3))                  # explore: try a random chest
        else:
            a = int(np.argmax(values))                # exploit: best chest so far
        r = open_chest(a)                             # feedback
        counts[a] += 1
        values[a] += (r - values[a]) / counts[a]      # learn: running average
        loot += r
    return loot, counts

for eps in [0.0, 0.01, 0.1, 0.3, 1.0]:
    runs = [epsilon_greedy(epsilon=eps) for _ in range(20)]
    loot = np.mean([l for l, _ in runs])
    share = np.mean([c[2] / c.sum() for _, c in runs])
    print(f"epsilon={eps:<5} avg rare loot={loot:6.0f}   opens on golden chest={share:.0%}")

epsilon=0.0   avg rare loot=   398   opens on golden chest=0%
epsilon=0.01  avg rare loot=   888   opens on golden chest=73%
epsilon=0.1   avg rare loot=   944   opens on golden chest=88%
epsilon=0.3   avg rare loot=   904   opens on golden chest=79%
epsilon=1.0   avg rare loot=   708   opens on golden chest=33%


**What to notice**
- **ε = 0** (never explore): opens the wooden chest first, gets some loot, and never tries the others. **0%** of opens on the golden chest.
- **ε = 1** (always explore): just random.
- A little exploration wins, but you had to tune ε by hand.

### 1b · Thompson sampling
Keep a **belief** about each chest, built from its wins and losses. Each turn, imagine a plausible drop rate for every chest by sampling from its belief, and open the best imagined one. Chests you're unsure about sometimes get lucky guesses and get tried. **Uncertainty drives exploration**, with no knob to tune.

In [2]:
import numpy as np
rng = np.random.default_rng(42)
TRUE_DROP_RATES = np.array([0.20, 0.35, 0.50])
def open_chest(i): return int(rng.random() < TRUE_DROP_RATES[i])

def thompson(n_opens=2000):
    wins = np.ones(3); losses = np.ones(3); loot = 0
    for t in range(n_opens):
        guesses = rng.beta(wins, losses)      # imagine a plausible drop rate for each chest
        a = int(np.argmax(guesses))           # open the most promising one
        r = open_chest(a)
        wins[a] += r; losses[a] += 1 - r      # update beliefs
        loot += r
    return loot, wins + losses - 2

runs = [thompson() for _ in range(20)]
print(f"Thompson sampling: avg rare loot={np.mean([l for l,_ in runs]):.0f}   "
      f"opens on golden chest={np.mean([c[2]/c.sum() for _,c in runs]):.0%}")
print(f"(Perfect play, always golden: {0.5*2000:.0f})")

Thompson sampling: avg rare loot=984   opens on golden chest=95%
(Perfect play, always golden: 1000)


** Try it**
1. Make two chests nearly equal (0.48 and 0.50). Which strategy copes better?
2. Halfway through, make the wooden chest the best. Does the agent notice? *(Thompson sampling gets slow to notice because old evidence piles up. The fix: gradually forget old wins and losses.)*

---
# Level 2 · Dungeon Escape 

Now decisions are connected. Where the hero stands affects what they should do next, and the big reward (escaping) only arrives **at the end**.

- Every step: **−1** (hunger)
- Stepping in lava `L`: **−10**
- Reaching the exit `E`: **+20**

**Q-learning** keeps a score for every square and direction: *"if I'm here and I move right, how much total reward will I get from now on?"*

> new score = old score + α × ( reward now + γ × best score from the next square − old score )

- **α (alpha)**: how far to move toward the new evidence
- **γ (gamma)**: how much the future matters

This is the **Bellman equation**. Over many runs, the value of the exit flows backwards, square by square, until the very first move "knows" which way to go.

In [3]:
import numpy as np
rng = np.random.default_rng(0)

# The hero (H) must escape the dungeon to the exit (E).
# 'L' is lava: stepping in hurts badly. Every step costs a little (hunger).
DUNGEON = [
    "H...L",
    ".LL..",
    "...L.",
    "L.L..",
    "....E",
]
SIZE = 5
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # up, down, left, right
START, EXIT = (0, 0), (4, 4)

def step(pos, a):
    r, c = pos
    nr = min(max(r + MOVES[a][0], 0), SIZE - 1)
    nc = min(max(c + MOVES[a][1], 0), SIZE - 1)
    if (nr, nc) == EXIT:
        return (nr, nc), 20.0, True                # escaped!
    if DUNGEON[nr][nc] == "L":
        return (nr, nc), -10.0, False              # ouch, lava
    return (nr, nc), -1.0, False                   # one step closer to starving

Q = np.zeros((SIZE, SIZE, 4))                      # the hero's value estimates
alpha, gamma, epsilon = 0.1, 0.95, 0.2
scores = []

for episode in range(800):
    pos, done, score, steps = START, False, 0.0, 0
    while not done and steps < 100:
        a = int(rng.integers(4)) if rng.random() < epsilon else int(np.argmax(Q[pos]))
        nxt, r, done = step(pos, a)
        target = r + (0 if done else gamma * np.max(Q[nxt]))    # Bellman target
        Q[pos][a] += alpha * (target - Q[pos][a])               # nudge toward it
        pos, score, steps = nxt, score + r, steps + 1
    scores.append(score)

for lo, hi in [(0, 50), (100, 150), (750, 800)]:
    print(f"episodes {lo:>3}-{hi:<3} average score: {np.mean(scores[lo:hi]):6.1f}")

pos, path = START, [START]
for _ in range(30):
    pos, _, done = step(pos, int(np.argmax(Q[pos])))
    path.append(pos)
    if done: break
print("\nThe hero's learned escape route:")
for r in range(SIZE):
    print(" ".join("*" if (r, c) in path and DUNGEON[r][c] not in "HE" else DUNGEON[r][c] for c in range(SIZE)))
print(f"\n{len(path)-1} moves, lava touched: {sum(DUNGEON[r][c]=='L' for r, c in path)}")

episodes   0-50  average score:  -15.1
episodes 100-150 average score:    5.6
episodes 750-800 average score:    6.0

The hero's learned escape route:
H . . . L
* L L . .
* * . L .
L * L . .
. * * * E

8 moves, lava touched: 0


**What to notice**
- Early runs average around **−15**: the hero wanders into lava.
- By the end, it found an **8-move escape that never touches lava**. Nobody gave it a map.
- Training scores stay below the best possible because the hero still explores 20% of the time.

** Try it**
1. Set `gamma = 0.1`. The hero becomes short-sighted. Predict what happens, then run it.
2. Make lava cost only `−2`. Does the hero start taking shortcuts through it?
3. Add a treasure square worth `+5`. Does the hero detour to grab it?

---
# Level 3 · The Self-Play Champion ♟️

This is the purest form of a self-improving agent. There's **no teacher, no dataset, no human games**. A single brain plays tic-tac-toe **against itself**, thousands of times.

After each game:
- every position the **winner** created becomes a little more valuable
- every position the **loser** created becomes a little less valuable
- draws pull positions toward neutral

When choosing a move, the brain picks the one leading to its most valuable position. Because it plays **both** sides, every game makes its opponent stronger too. Each version of the brain trains against a slightly better version of itself.

This is the same idea as **AlphaZero** (DeepMind, 2017), which learned chess, shogi and Go to superhuman level from self-play alone, with a deep neural network in place of our lookup table.

In [4]:
import numpy as np
from collections import defaultdict
rng = np.random.default_rng(3)

# Tic-tac-toe. The board is a tuple of 9 cells: 1 = X, -1 = O, 0 = empty.
LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def winner(b):
    for i, j, k in LINES:
        if b[i] != 0 and b[i] == b[j] == b[k]:
            return b[i]
    return 0 if 0 in b else "draw"

# ONE brain plays BOTH sides. It scores positions from the point of view of
# whoever just moved, so everything it learns as X also helps it as O.
value = defaultdict(float)            # value[position] = how good this is for the player who just moved
visits = defaultdict(int)

def as_mover(board, player):
    return tuple(c * player for c in board)   # flip the board so "my" pieces are always +1

def choose(board, player, explore=0.1):
    moves = [i for i in range(9) if board[i] == 0]
    if rng.random() < explore:
        return int(rng.choice(moves))
    scores = []
    for m in moves:
        b = list(board); b[m] = player
        scores.append(value[as_mover(b, player)])
    best = max(scores)
    return int(rng.choice([m for m, s in zip(moves, scores) if s == best]))

def self_play_game(explore=0.1):
    board, player, history = [0]*9, 1, []
    while True:
        m = choose(board, player, explore)
        board[m] = player
        history.append((as_mover(board, player), player))
        result = winner(board)
        if result != 0:
            break
        player = -player
    # Learn: every position the winner created moves toward +1, the loser's toward -1, draws toward 0
    for pos, p in history:
        target = 0.0 if result == "draw" else (1.0 if result == p else -1.0)
        visits[pos] += 1
        value[pos] += (target - value[pos]) / visits[pos]

def random_move(board):
    return int(rng.choice([i for i in range(9) if board[i] == 0]))

def match_vs_random(games=400):
    """Test the brain (no exploring) against a random player, half as X and half as O."""
    wins = draws = losses = 0
    for g in range(games):
        me = 1 if g % 2 == 0 else -1
        board, player = [0]*9, 1
        while True:
            m = choose(board, player, explore=0) if player == me else random_move(board)
            board[m] = player
            result = winner(board)
            if result != 0: break
            player = -player
        if result == "draw": draws += 1
        elif result == me: wins += 1
        else: losses += 1
    return wins / games, draws / games, losses / games

print("Games of self-play  |  vs random player:  win   draw   loss")
played = 0
for checkpoint in [0, 100, 1000, 5000, 20000]:
    while played < checkpoint:
        self_play_game(); played += 1
    w, d, l = match_vs_random()
    print(f"{checkpoint:>18}  |                    {w:4.0%}  {d:5.0%}  {l:5.0%}")
print(f"\nPositions the brain has learned about: {len(value)}")

Games of self-play  |  vs random player:  win   draw   loss
                 0  |                     44%    14%    42%
               100  |                     54%    12%    34%
              1000  |                     74%     9%    17%
              5000  |                     80%    10%    10%
             20000  |                     88%     7%     5%

Positions the brain has learned about: 5270


**What to notice**
- With **0 games** of practice, the brain plays like a random player.
- After **20,000 games against itself**, it wins most games against a random player and rarely loses.
- It never saw a single human game or strategy guide.

### Can you beat it?
Run the next cell and play as **X**. Squares are numbered 1–9, left to right, top to bottom.

In [ ]:
def show(board):
    s = ["X" if c == 1 else "O" if c == -1 else str(i + 1) for i, c in enumerate(board)]
    print(f"\n {s[0]} | {s[1]} | {s[2]}\n---+---+---\n {s[3]} | {s[4]} | {s[5]}\n---+---+---\n {s[6]} | {s[7]} | {s[8]}\n")

def play_against_brain():
    """You are X and move first. Type a square number 1-9."""
    board, player = [0] * 9, 1
    while True:
        show(board)
        if player == 1:
            entry = input("Your move (1-9): ").strip()
            m = int(entry) - 1 if entry.isdigit() else -1
            if not 0 <= m < 9 or board[m] != 0:
                print("That square isn't free. Try again."); continue
        else:
            m = choose(board, player, explore=0)
            print(f"The brain plays {m + 1}")
        board[m] = player
        result = winner(board)
        if result != 0:
            show(board)
            print("Draw!" if result == "draw" else "You win!" if result == 1 else "The brain wins!")
            return
        player = -player

play_against_brain()


 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

That square isn't free. Try again.

 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

That square isn't free. Try again.

 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

That square isn't free. Try again.

 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

That square isn't free. Try again.

 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9


 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | X

The brain plays 1

 O | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | X

That square isn't free. Try again.

 O | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | X

That square isn't free. Try again.

 O | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | X

That square isn't free. Try again.

 O | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | X

That square isn't free. Try again.

 O | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | X



**🧪 Try it**
1. Train for only 1,000 games and play against it. Can you find its weak spots?
2. Set `explore=0` during self-play. Why does it learn worse? *(Without exploring, it keeps replaying the same games and never discovers better moves.)*
3. Perfect tic-tac-toe never loses. How many games of self-play does it take to get the loss rate to 0%?

---
# Level 4 · The Wizard's Spellbook

Now the bridge to modern AI. A language model is a **policy**: given the prompt and the text so far, it picks the next token. Training it with RL means: *make good answers more likely, bad answers less likely.*

### The task
Prompt: *"Monster has 70 HP."* Our tiny model writes up to 3 spells:

| Spell | Damage |
|---|---|
| Spark | 5 |
| Fireball | 10 |
| Lightning | 20 |
| Meteor | 50 |

**Mana is precious.** The perfect answer defeats the monster with **exactly** its HP: no overkill, no wasted mana. Our reward is **verifiable**: code checks the total damage. No human needed.

### GRPO: Group Relative Policy Optimization
The method used to train DeepSeek-R1 and many open reasoning models. For each prompt:
1. Sample a **group** of answers (here, 8)
2. Score each with the reward
3. Compute each answer's **advantage**: `(reward − group mean) / group std`
4. Make better-than-average answers more likely; worse ones less likely

The group *is* the baseline. No separate "critic" model needed.

> Simplification: real GRPO also limits how big each update can be, and penalises drifting too far from the original model. We skip both so the core idea stays visible.

In [ ]:
import numpy as np
from collections import defaultdict
rng = np.random.default_rng(1)

# A tiny "language model" for a game. Prompt: "Monster has 70 HP".
# The model writes up to 3 tokens: spells to cast, or STOP.
VOCAB = ["Spark", "Fireball", "Lightning", "Meteor", "STOP"]
DAMAGE = {"Spark": 5, "Fireball": 10, "Lightning": 20, "Meteor": 50}
MONSTERS_HP = [15, 25, 30, 40, 60, 70, 80, 110, 120]
MAX_SPELLS = 3

class TinyPolicy:
    """Next-token probabilities, conditioned on the prompt and everything written so far,
    exactly like an LLM. Here it's a lookup table instead of a transformer."""
    def __init__(self):
        self.logits = defaultdict(lambda: np.zeros(len(VOCAB)))

    def probs(self, hp, prefix):
        z = self.logits[(hp, tuple(prefix))]
        e = np.exp(z - z.max()); return e / e.sum()

    def generate(self, hp, greedy=False):
        out = []
        while len(out) < MAX_SPELLS:
            p = self.probs(hp, out)
            tok = VOCAB[int(np.argmax(p))] if greedy else VOCAB[rng.choice(len(VOCAB), p=p)]
            if tok == "STOP": break
            out.append(tok)
        return out

def damage(spells):
    return sum(DAMAGE[s] for s in spells)

def reward_exact(hp, spells):          # verifiable: defeated with zero wasted mana?
    return 1.0 if damage(spells) == hp else 0.0

def evaluate(policy, reward_fn, samples=20):
    trained = np.mean([reward_fn(h, policy.generate(h)) for h in MONSTERS_HP for _ in range(samples)])
    exact = np.mean([reward_exact(h, policy.generate(h)) for h in MONSTERS_HP for _ in range(samples)])
    return trained, exact

def grpo_train(reward_fn, steps=60, group_size=8, lr=0.3, label=""):
    policy = TinyPolicy()
    for step in range(steps + 1):
        if step in (0, 10, 20, 40, 60):
            trained, exact = evaluate(policy, reward_fn)
            print(f"{label} step {step:>2}   reward it was trained on: {trained:4.0%}   perfect kills: {exact:4.0%}")
        for hp in MONSTERS_HP:
            # 1. Sample a GROUP of answers to the same prompt
            group = [policy.generate(hp) for _ in range(group_size)]
            rewards = np.array([reward_fn(hp, g) for g in group])
            # 2. Advantage = how much better than the group average?
            adv = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
            # 3. Make better-than-average answers more likely, worse ones less likely
            for spells, a in zip(group, adv):
                if a == 0: continue
                tokens = spells + (["STOP"] if len(spells) < MAX_SPELLS else [])
                for i, tok in enumerate(tokens):
                    grad = -policy.probs(hp, spells[:i])
                    grad[VOCAB.index(tok)] += 1.0             # d log p(token) / d logits
                    policy.logits[(hp, tuple(spells[:i]))] += lr * a * grad / len(tokens)
    return policy

print("=== Reward: defeat the monster with EXACTLY its HP (no wasted mana) ===")
wizard = grpo_train(reward_exact, label="[exact]")
for hp in [70, 120]:
    print(f"  Monster with {hp} HP: {' + '.join(wizard.generate(hp, greedy=True))}")

print("\n=== Lazy reward: 'was the monster defeated?' ===")
def reward_defeated(hp, spells):
    return 1.0 if damage(spells) >= hp else 0.0

lazy = grpo_train(reward_defeated, label="[defeated]")
for hp in [15, 70]:
    s = lazy.generate(hp, greedy=True)
    print(f"  Monster with {hp} HP: {' + '.join(s)} = {damage(s)} damage  ({damage(s) - hp} wasted)")

**What to notice**

**Run 1:** with the exact reward, the wizard goes from almost always wasteful to almost always perfect. Same recipe as today's reasoning models, just tiny.

**Run 2:** we got lazy and rewarded *"was the monster defeated?"*. The wizard scores nearly **100%** on that reward... by **spamming Meteor at everything**. A 15 HP rat gets a 50-damage Meteor. It learned exactly what we measured, not what we meant.

This is **reward hacking**, and real game AI does it all the time. Researchers once trained an agent to win a boat race, rewarded by points. It learned to spin in circles collecting respawning bonuses forever, and never finished the race.

> **Your reward function is your game design.** The agent will take it literally.

**Try it**
1. Set `group_size=2`. Learning gets noisier. Why?
2. When every answer in a group gets the same reward, the advantage is zero and nothing is learned. Why is that a problem for very hard monsters?
3. Fix the lazy reward: full reward for an exact kill, partial reward that **shrinks** as overkill grows.

---
# Level 5 · The Game-Dev Agent 

GRPO changes a model's **weights**. But there's a cheaper kind of self-improvement you can use with any model today: let the agent learn from its own failures and keep the lessons in **memory**. This is called **Reflexion**.

Our agent must write `game_status(board)` for tic-tac-toe: return the winner, `'draw'`, or `'ongoing'`. Hidden tests are the reward.

The loop:
1. **Act**: the model writes code
2. **Feedback**: hidden tests run; reward = fraction passed
3. **Reflect**: the model writes a one-sentence lesson about what went wrong
4. **Update**: the lesson goes into memory for the next attempt

**Choose a backend**
- `"mock"`: scripted replies, runs offline, no key needed (default)
- `"gemini"`: `pip install google-genai`, add `GEMINI_API_KEY` in Colab's 🔑 Secrets panel, and set `GEMINI_MODEL` to the current Flash model if the default has changed
- `"claude"`: `pip install anthropic`, set `ANTHROPIC_API_KEY`

In [ ]:
import os
os.environ["LLM_BACKEND"] = "mock"     # change to "gemini" or "claude"

# In Colab, load your key from Secrets:
# from google.colab import userdata
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
import os, re, subprocess, sys, tempfile, textwrap

BACKEND = os.environ.get("LLM_BACKEND", "mock")     # "mock", "gemini" or "claude"

def call_llm(prompt):
    if BACKEND == "gemini":
        from google import genai                     # pip install google-genai
        client = genai.Client()                      # reads GEMINI_API_KEY
        model = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
        return client.models.generate_content(model=model, contents=prompt).text
    if BACKEND == "claude":
        import anthropic                             # pip install anthropic
        client = anthropic.Anthropic()               # reads ANTHROPIC_API_KEY
        msg = client.messages.create(model=os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5"),
                                     max_tokens=1500, messages=[{"role": "user", "content": prompt}])
        return "".join(b.text for b in msg.content if b.type == "text")
    return next(MOCK_REPLIES)

MOCK_REPLIES = iter([
    "```python\ndef game_status(board):\n    for r in range(3):\n        row = board[r*3:r*3+3]\n"
    "        if row[0] != ' ' and row[0] == row[1] == row[2]:\n            return row[0]\n    return 'draw'\n```",
    "I only checked rows. A player can also win with a column or a diagonal.",
    "```python\ndef game_status(board):\n    lines = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]\n"
    "    for a, b, c in lines:\n        if board[a] != ' ' and board[a] == board[b] == board[c]:\n            return board[a]\n"
    "    return 'draw'\n```",
    "If nobody has won but there are still empty squares, the game is not a draw: it is still ongoing.",
    "```python\ndef game_status(board):\n    lines = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]\n"
    "    for a, b, c in lines:\n        if board[a] != ' ' and board[a] == board[b] == board[c]:\n            return board[a]\n"
    "    return 'ongoing' if ' ' in board else 'draw'\n```",
])

TASK = ("Write a Python function `game_status(board: str) -> str` for tic-tac-toe. "
        "`board` is a 9-character string read left to right, top to bottom, using 'X', 'O' and ' '. "
        "Return 'X' or 'O' if that player has won, 'draw' if the board is full with no winner, "
        "otherwise 'ongoing'. Return only the code in a single ```python block.")

# The hidden tests ARE the reward. The agent only sees which ones failed.
TESTS = textwrap.dedent('''
    from solution import game_status
    cases = [
        ("XXXOO    ", "X"),   # top row
        ("XO XO X  ", "X"),   # left column
        ("OX XO  XO", "O"),   # diagonal
        ("X OXO O X", "O"),   # anti-diagonal
        ("XOXXOOOXX", "draw"),
        ("X O      ", "ongoing"),
        ("         ", "ongoing"),
        (" O XOX O ", "O"),   # middle column
    ]
    failed = []
    for board, expected in cases:
        try: got = game_status(board)
        except Exception as e: got = f"raised {type(e).__name__}"
        if got != expected: failed.append(f"game_status({board!r}) returned {got!r}, expected {expected!r}")
    print(f"PASSED {len(cases)-len(failed)}/{len(cases)}")
    for f in failed: print("FAIL:", f)
''')

def run_tests(code):
    """The environment: run the agent's code against hidden tests. Returns (reward, feedback)."""
    with tempfile.TemporaryDirectory() as d:
        open(os.path.join(d, "solution.py"), "w").write(code)
        open(os.path.join(d, "tests.py"), "w").write(TESTS)
        try:
            out = subprocess.run([sys.executable, "tests.py"], cwd=d, capture_output=True,
                                 text=True, timeout=10).stdout.strip()
        except subprocess.TimeoutExpired:
            out = "PASSED 0/8\nFAIL: timed out"
    m = re.match(r"PASSED (\d+)/(\d+)", out)
    reward = int(m.group(1)) / int(m.group(2)) if m else 0.0
    return reward, out

lessons = []                                         # the agent's memory
for attempt in range(1, 5):
    memory = "\n".join(f"- {l}" for l in lessons) or "(none yet)"
    reply = call_llm(f"{TASK}\n\nLessons from your previous attempts:\n{memory}")          # ACT
    code = (re.search(r"```python\s*(.*?)```", reply, re.S) or [None, reply])[1]
    reward, feedback = run_tests(code)                                                       # FEEDBACK
    print(f"Attempt {attempt}: reward = {reward:.2f}")
    if reward == 1.0:
        print("All tests pass.\n"); print(code); break
    for line in feedback.splitlines()[1:]: print("  ", line)
    lesson = call_llm(f"This code:\n```python\n{code}\n```\nfailed these tests:\n{feedback}\n\n"   # REFLECT
                      "In one or two sentences, state the general lesson to remember. No code.").strip()
    lessons.append(lesson)                                                                   # UPDATE memory
    print(f"   Lesson stored: {lesson}\n")

**What to notice**
- Same model, same weights. The only thing that improved was the `lessons` list.
- The reward is objective: tests pass or fail. That's what keeps the loop honest.

>  This runs model-written code with only a timeout as protection. Fine for a workshop; use a proper sandbox (like a container) anywhere it matters.

**Try it**
1. Remove the reflection step and just retry. Does it still improve?
2. Delete the column and diagonal tests. Does the agent "pass" with broken code? (Reward hacking again.)
3. Save `lessons` to a file so the agent remembers across runs.
4. Plug the agent's working `game_status` into the Level 3 game.

---
# Level 6 · Boss Level: GRPO on a real language model 👾 *(optional, needs a GPU)*

The Wizard's Spellbook again, but now with a real LLM, using Hugging Face's **TRL** library. In Colab: **Runtime → Change runtime type → T4 GPU**.

- Same spells, same exact-damage reward
- `num_generations=8` is the group size, just like Level 4

**Model choice:** this uses a small open model (`Qwen/Qwen2.5-0.5B-Instruct`) that fits on a free GPU. To use Gemma, accept its licence on Hugging Face, log in, and swap in a small instruction-tuned Gemma checkpoint.

> TRL's API evolves. If a setting name errors, check the GRPOTrainer docs for your installed version. Training takes a while on a free GPU; lower `max_steps` for a quick demo.

In [ ]:
%pip install -q trl datasets

In [ ]:
import random, re
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

MONSTERS_HP = [15, 25, 30, 40, 60, 70, 80, 110, 120]
DAMAGE = {"spark": 5, "fireball": 10, "lightning": 20, "meteor": 50}
PROMPT = ("A monster has {hp} HP. Your spells: Spark (5 damage), Fireball (10), "
          "Lightning (20), Meteor (50). Defeat it using at most 3 spells, dealing EXACTLY "
          "{hp} damage so no mana is wasted. Answer only with spell names joined by +, "
          "for example: Fireball + Lightning")

dataset = Dataset.from_list(
    [{"prompt": PROMPT.format(hp=hp), "hp": hp} for hp in random.choices(MONSTERS_HP, k=512)]
)

def exact_kill_reward(completions, hp, **kwargs):
    """Verifiable reward: 1.0 if the spells deal exactly the monster's HP, else 0.0."""
    rewards = []
    for text, target in zip(completions, hp):
        spells = re.findall(r"\b(spark|fireball|lightning|meteor)\b", text.lower())
        ok = spells and len(spells) <= 3 and sum(DAMAGE[s] for s in spells) == target
        rewards.append(1.0 if ok else 0.0)
    return rewards

args = GRPOConfig(
    output_dir="wizard-grpo",
    num_generations=8,              # the GROUP size
    per_device_train_batch_size=8,
    max_completion_length=32,
    learning_rate=1e-5,
    max_steps=100,
    logging_steps=5,
)

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=exact_kill_reward,
    args=args,
    train_dataset=dataset,
)
trainer.train()

Watch the **reward** column in the training log climb. That's a real model getting better at a task purely from a reward function you wrote.

---
# Game over: what you built

| Level | Where feedback came from | What changed |
|---|---|---|
| 1 · Loot Chests | Loot or junk | Beliefs about each chest |
| 2 · Dungeon Escape | Reward now + estimated future | Value of each square and move |
| 3 · Self-Play Champion | Winning or losing against itself | Value of each board position |
| 4 · Wizard's Spellbook | Automatic checker, compared within a group | Model weights |
| 5 · Game-Dev Agent | Tests + self-critique | Memory |
| 6 · Boss Level | Automatic checker | Real LLM weights |

### Five questions before you build a self-improving agent
1. What's the reward, and can code check it?
2. How could the agent score well while doing something you don't want? *(Meteor spam.)*
3. Where should improvement live: memory, tools, or weights? *(Start with memory: cheap, readable, reversible.)*
4. How much exploration can you afford?
5. Who's watching? Log every decision and every reward.

> **Agents don't improve on their own. They improve on the loop you build for them.**

### Further reading
- Sutton & Barto, *Reinforcement Learning: An Introduction* (free online)
- Silver et al., AlphaZero, *Science*, 2018
- Shao et al., *DeepSeekMath* (introduces GRPO), 2024
- DeepSeek-AI, *DeepSeek-R1*, 2025
- Shinn et al., *Reflexion*, 2023
- Amodei & Clark, *Faulty Reward Functions in the Wild* (the boat race), OpenAI, 2016
- Hugging Face TRL documentation: GRPOTrainer